# Visual HMTL 训练 (Colab)

**模型**: EfficientNet-B2 + HMTL 多任务头

**数据集**: AffectNet-like (28,175 样本)

## 1. 环境检查

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. 挂载 Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# ========== 修改这里的路径 ==========
DRIVE_PATH = "/content/drive/MyDrive"
# ===================================

LABELS_CSV = f"{DRIVE_PATH}/visual_hmtl_labels.csv"
DATA_DIR = f"{DRIVE_PATH}/visual_data"  # 图片文件夹
MODEL_SAVE_PATH = f"{DRIVE_PATH}/visual_hmtl_best.pt"

print(f"标签文件存在: {os.path.exists(LABELS_CSV)}")

## 3. 定义模型

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models

class VisualHMTLClassifier(nn.Module):
    def __init__(self, dropout=0.3):
        super().__init__()
        
        # EfficientNet-B2 骨干
        self.backbone = models.efficientnet_b2(weights=models.EfficientNet_B2_Weights.IMAGENET1K_V1)
        feature_dim = self.backbone.classifier[1].in_features  # 1408
        self.backbone.classifier = nn.Identity()
        
        # 共享层
        self.shared_fc = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(feature_dim, 512),
            nn.ReLU(),
            nn.Dropout(dropout * 0.5)
        )
        
        # 7类情绪
        self.classifier_7 = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(dropout * 0.5),
            nn.Linear(256, 7)
        )
        
        # 4类
        self.classifier_4 = nn.Sequential(
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Dropout(dropout * 0.5),
            nn.Linear(128, 4)
        )
        
        # 3类
        self.classifier_3 = nn.Sequential(
            nn.Linear(512, 64),
            nn.ReLU(),
            nn.Dropout(dropout * 0.5),
            nn.Linear(64, 3)
        )
        
        # Arousal
        self.regressor_A = nn.Sequential(
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Linear(128, 1),
            nn.Sigmoid()
        )
        
        # Valence
        self.regressor_V = nn.Sequential(
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Linear(128, 1),
            nn.Tanh()
        )
    
    def forward(self, x):
        features = self.backbone(x)
        shared = self.shared_fc(features)
        
        return {
            'label_7_logits': self.classifier_7(shared),
            'label_4_logits': self.classifier_4(shared),
            'label_3_logits': self.classifier_3(shared),
            'arousal': self.regressor_A(shared).squeeze(-1),
            'valence': self.regressor_V(shared).squeeze(-1)
        }

print("模型定义完成!")

## 4. 定义数据集

In [ ]:
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import pandas as pd
import numpy as np

class VisualEmotionDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        # 加载图片
        img_path = row['image_path']
        try:
            image = Image.open(img_path).convert('RGB')
            if self.transform:
                image = self.transform(image)
        except Exception as e:
            # 返回黑图
            image = torch.zeros(3, 224, 224)
        
        return {
            'image': image,
            'label_7': torch.tensor(row['label_7'], dtype=torch.long),
            'label_4': torch.tensor(row['label_4'], dtype=torch.long),
            'label_3': torch.tensor(row['label_3'], dtype=torch.long),
            'arousal': torch.tensor(row['arousal'], dtype=torch.float),
            'valence': torch.tensor(row['valence'], dtype=torch.float),
        }


# 数据增强
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

print("数据集定义完成!")

## 5. 定义损失函数

In [ ]:
class HMTLVisualLoss(nn.Module):
    def __init__(self, class_weights_7=None, class_weights_4=None):
        super().__init__()
        self.ce_loss_7 = nn.CrossEntropyLoss(weight=class_weights_7)
        self.ce_loss_4 = nn.CrossEntropyLoss(weight=class_weights_4)
        self.ce_loss_3 = nn.CrossEntropyLoss()
        self.mse_loss = nn.MSELoss()
    
    def forward(self, outputs, targets):
        loss_7 = self.ce_loss_7(outputs['label_7_logits'], targets['label_7'])
        loss_4 = self.ce_loss_4(outputs['label_4_logits'], targets['label_4'])
        loss_3 = self.ce_loss_3(outputs['label_3_logits'], targets['label_3'])
        loss_A = self.mse_loss(outputs['arousal'], targets['arousal'])
        loss_V = self.mse_loss(outputs['valence'], targets['valence'])
        
        # 权重: 7类主导
        total_loss = 1.0 * loss_7 + 0.5 * loss_4 + 0.3 * loss_3 + 0.3 * loss_A + 0.3 * loss_V
        
        return {
            'total_loss': total_loss,
            'loss_7': loss_7.item(),
            'loss_4': loss_4.item(),
        }

print("损失函数定义完成!")

## 6. 加载数据

In [ ]:
from sklearn.model_selection import train_test_split

# 加载标签
print(f"加载标签: {LABELS_CSV}")
label_df = pd.read_csv(LABELS_CSV)
print(f"总样本数: {len(label_df)}")

# 修正路径 (如果需要)
# label_df['image_path'] = label_df['image_path'].str.replace('d:\\bigcreate\\visual_data_temp', DATA_DIR)

# 显示7类分布
print("\n7类标签分布:")
emotion_names = {0: '愤怒', 1: '焦虑', 2: '快乐', 3: '悲伤', 4: '失望', 5: '支持', 6: '平静'}
for label_id in sorted(label_df['label_7'].unique()):
    count = len(label_df[label_df['label_7'] == label_id])
    print(f"  [{label_id}] {emotion_names.get(label_id, '未知')}: {count} ({count/len(label_df)*100:.1f}%)")

# 划分数据集
train_df, test_df = train_test_split(label_df, test_size=0.2, random_state=42, stratify=label_df['label_7'])
print(f"\n训练集: {len(train_df)}, 测试集: {len(test_df)}")

In [ ]:
# 创建数据加载器
BATCH_SIZE = 32

train_dataset = VisualEmotionDataset(train_df, transform=train_transform)
test_dataset = VisualEmotionDataset(test_df, transform=test_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"训练批次: {len(train_loader)}, 测试批次: {len(test_loader)}")

## 7. 创建模型和优化器

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"使用设备: {device}")

# 创建模型
model = VisualHMTLClassifier().to(device)

# 7类权重 (根据分布)
# [愤怒:3608, 焦虑:3043, 快乐:8952, 悲伤:6467, 失望:3244, 支持:0, 平静:2861]
CLASS_WEIGHTS_7 = torch.tensor([
    28175 / (6 * 3608),   # 愤怒
    28175 / (6 * 3043),   # 焦虑
    28175 / (6 * 8952),   # 快乐
    28175 / (6 * 6467),   # 悲伤
    28175 / (6 * 3244),   # 失望
    0.0,                   # 支持 (无数据)
    28175 / (6 * 2861),   # 平静
]).to(device)

CLASS_WEIGHTS_4 = torch.tensor([
    28175 / (4 * 8952),   # 积极
    28175 / (4 * 6651),   # 激活消极
    28175 / (4 * 9711),   # 非激活消极
    28175 / (4 * 2861),   # 平静
]).to(device)

# 损失函数
criterion = HMTLVisualLoss(
    class_weights_7=CLASS_WEIGHTS_7,
    class_weights_4=CLASS_WEIGHTS_4
)

# 分层学习率
LEARNING_RATE = 1e-4

backbone_params = list(model.backbone.parameters())
head_params = (list(model.shared_fc.parameters()) +
               list(model.classifier_7.parameters()) +
               list(model.classifier_4.parameters()) +
               list(model.classifier_3.parameters()) +
               list(model.regressor_A.parameters()) +
               list(model.regressor_V.parameters()))

optimizer = torch.optim.AdamW([
    {'params': backbone_params, 'lr': LEARNING_RATE * 0.1},  # 骨干网络学习率低
    {'params': head_params, 'lr': LEARNING_RATE}
])

# 学习率调度器
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)

print("模型和优化器创建完成!")

## 8. 训练

In [ ]:
from tqdm import tqdm

def evaluate(model, dataloader, device):
    model.eval()
    correct_7 = 0
    correct_4 = 0
    total = 0
    
    with torch.no_grad():
        for batch in dataloader:
            images = batch['image'].to(device)
            targets = {k: v.to(device) for k, v in batch.items() if k != 'image'}
            
            outputs = model(images)
            
            pred_7 = outputs['label_7_logits'].argmax(dim=1)
            pred_4 = outputs['label_4_logits'].argmax(dim=1)
            
            correct_7 += (pred_7 == targets['label_7']).sum().item()
            correct_4 += (pred_4 == targets['label_4']).sum().item()
            total += targets['label_7'].size(0)
    
    return {
        'acc_7': correct_7 / total if total > 0 else 0,
        'acc_4': correct_4 / total if total > 0 else 0,
    }


NUM_EPOCHS = 10
best_acc_7 = 0

print("="*60)
print("开始训练 Visual HMTL")
print(f"Epochs: {NUM_EPOCHS}, Batch Size: {BATCH_SIZE}")
print("="*60)

for epoch in range(NUM_EPOCHS):
    model.train()
    epoch_loss = 0
    
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}")
    for batch in pbar:
        images = batch['image'].to(device)
        targets = {k: v.to(device) for k, v in batch.items() if k != 'image'}
        
        optimizer.zero_grad()
        outputs = model(images)
        loss_dict = criterion(outputs, targets)
        
        loss_dict['total_loss'].backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        epoch_loss += loss_dict['total_loss'].item()
        pbar.set_postfix({'loss': f"{loss_dict['total_loss'].item():.4f}"})
    
    scheduler.step()
    
    # 评估
    eval_result = evaluate(model, test_loader, device)
    
    print(f"Epoch {epoch+1}: Loss={epoch_loss/len(train_loader):.4f}, "
          f"Acc_7={eval_result['acc_7']*100:.2f}%, "
          f"Acc_4={eval_result['acc_4']*100:.2f}%")
    
    # 保存最佳模型
    if eval_result['acc_7'] > best_acc_7:
        best_acc_7 = eval_result['acc_7']
        torch.save(model.state_dict(), MODEL_SAVE_PATH)
        print(f"  -> 保存最佳模型: {best_acc_7*100:.2f}%")

print("\n" + "="*60)
print(f"训练完成! 最佳 7类准确率: {best_acc_7*100:.2f}%")
print(f"模型保存到: {MODEL_SAVE_PATH}")
print("="*60)

## 9. 测试模型输出格式

In [ ]:
# 加载最佳模型
model.load_state_dict(torch.load(MODEL_SAVE_PATH))
model.eval()

# 测试一个样本
test_batch = next(iter(test_loader))
images = test_batch['image'][:1].to(device)

with torch.no_grad():
    outputs = model(images)

print("模型输出格式:")
for key, value in outputs.items():
    print(f"  {key}: shape={value.shape}")

print("\n示例输出:")
pred_7 = outputs['label_7_logits'].argmax(dim=1).item()
print(f"  label_7 预测: {pred_7} ({emotion_names.get(pred_7, '未知')})")
print(f"  arousal: {outputs['arousal'][0].item():.4f}")
print(f"  valence: {outputs['valence'][0].item():.4f}")

print("\n✅ 输出格式与 Text/Audio HMTL 一致!")